In [ ]:
import pandas as pd
from tbdynamics.constants import (
    age_strata,
    organ_strata,
    compartments,

)
import nevergrad as ng
from estival.wrappers import nevergrad as eng
# Import our convenience wrapper
from estival.wrappers.nevergrad import optimize_model
from tbdynamics.calib_utils import get_bcm
from multiprocessing import cpu_count
from estival.sampling import tools as esamp
from estival.sampling.tools import SampleIterator, SampleTypes
from estival.utils.parallel import map_parallel
import pymc as pm
from estival.wrappers import pymc as epm
import arviz as az
import numpy as np

## Define Model

### Params and calibration targets

In [ ]:
bcm = get_bcm()

### Running Optimization

In [ ]:
def optimize_ng_with_idx(item):
    idx, sample = item
    opt = eng.optimize_model(bcm, budget=1000, opt_class=ng.optimizers.TwoPointsDE, suggested = sample, num_workers=4)
    rec= opt.minimize(1000)
    return idx, rec.value[1]

In [ ]:
lhs_samples = bcm.sample.lhs(16)

In [ ]:
lhs_lle = esamp.likelihood_extras_for_samples(lhs_samples, bcm)

In [ ]:
lhs_sorted = lhs_lle.sort_values("loglikelihood", ascending=False)

In [ ]:
best8 = lhs_samples[lhs_sorted.index].iloc[0:8]

In [ ]:
opt_samples_idx = map_parallel(optimize_ng_with_idx, best8.iterrows())

In [ ]:
lle_samps = esamp.likelihood_extras_for_samples(opt_samples_idx, bcm)

In [ ]:
best_opt_samps = bcm.sample.convert(opt_samples_idx)
best_opt_samps

In [ ]:
init_samps = best_opt_samps.iloc[0:8].convert("list_of_dicts")

In [ ]:
with pm.Model() as model:
    variables = epm.use_model(bcm)
    idata = pm.sample(step=[pm.DEMetropolisZ(variables)],draws=1000, chains=8, initvals=init_samps)

In [ ]:
az.summary(idata)

In [ ]:
lle = esamp.likelihood_extras_for_idata(idata, bcm)
lle["logposterior"].unstack(["chain"]).rolling(250).mean().plot()

In [ ]:
burnt_idata = idata.sel(draw=np.s_[200:])
sds = az.extract(burnt_idata, num_samples=500)

In [ ]:
samp_res = esamp.model_results_for_samples(sds,bcm)

In [ ]:
quantiles = esamp.quantiles_for_results(samp_res.results, (0.05,0.25,0.5,0.75,0.95))

In [ ]:
quantiles["notification"].plot()
bcm.targets["notification"].data.plot(style='.',color="black")

In [ ]:
quantiles["total_population"].plot()
bcm.targets["total_population"].data.plot(style='.',color="black")